In [ ]:
import pandas as pd

df = pd.read_csv("spam.csv", encoding="latin-1")
print(df.shape)
print(df.head())
print(df.columns.tolist())

In [ ]:
df = df[["v1", "v2"]].rename(columns={"v1": "label", "v2": "text"})
df["label"] = df["label"].map({"ham": 0, "spam": 1})
print(df["label"].value_counts())
print(df.head())

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-mpnet-base-v2")
X_emb = embedder.encode(df["text"].tolist(), show_progress_bar=True)
y = df["label"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X_emb, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick your PDF
pdf_path = list(uploaded.keys())[0]
print(pdf_path)

In [ ]:
!pip install -q pymupdf

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu

from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

import fitz, re, numpy as np, faiss, json
from sentence_transformers import SentenceTransformer

doc = fitz.open(pdf_path)
pages = []
for i, page in enumerate(doc):
    pages.append({"page": i + 1, "text": page.get_text()})

def clean_text(text, page_num, total_pages):
    text = text.replace(f"Page {page_num} of {total_pages}", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for p in pages:
    p["clean_text"] = clean_text(p["text"], p["page"], len(pages))

def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size]
        if chunk:
            chunks.append(" ".join(chunk))
        if i + chunk_size >= len(words):
            break
    return chunks

all_chunks = []
chunk_id = 0
for p in pages:
    for c in chunk_text(p["clean_text"]):
        all_chunks.append({"chunk_id": chunk_id, "page": p["page"], "text": c})
        chunk_id += 1

embedder = SentenceTransformer("all-mpnet-base-v2")
chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = embedder.encode(chunk_texts, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")
faiss.normalize_L2(chunk_embeddings)

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

faiss.write_index(index, "chunks.index")
with open("chunks_meta.json", "w") as f:
    json.dump(all_chunks, f)

print("done:", len(pages), "pages,", len(all_chunks), "chunks")

In [ ]:
def search(query, k=3):
    q_emb = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, k)
    for score, idx in zip(scores[0], idxs[0]):
        c = all_chunks[idx]
        print(f"[page {c['page']}] score={score:.3f}")
        print(c["text"][:200])
        print("---")

search("wave function probability interpretation")

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("OPENAI_API_KEY" in os.environ)

In [ ]:
import requests

def call_openai(prompt, model="gpt-4o-mini"):
    r = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
            "Content-Type": "application/json"
        },
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0
        }
    )
    data = r.json()
    if "error" in data:
        raise Exception(data["error"])
    return data["choices"][0]["message"]["content"]

def retrieve(query, k=4):
    q_emb = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, k)
    return [all_chunks[idx] for idx in idxs[0]]

def ask(query, k=4):
    chunks = retrieve(query, k)
    context = "\n\n".join(f"[page {c['page']}] {c['text']}" for c in chunks)
    prompt = f"""Answer the question using only the context below. If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {query}
Answer:"""
    answer = call_openai(prompt)
    return answer, chunks

In [ ]:
answer, chunks = ask("what is quantum mechanics")
print(answer)

In [ ]:
questions = [
    "What is quantum mechanics?",
    "Who first coined the term 'quantum mechanics'?",
    "What is the dual nature of light?",
    "How does quantum mechanics relate to Newtonian mechanics failing?",
    "What is the capital of France?"   # not in the PDF — should say "don't know"
]

for q in questions:
    answer, chunks = ask(q)
    print("Q:", q)
    print("A:", answer)
    print("Sources:", [c["page"] for c in chunks])
    print("=" * 80)

In [ ]:
def ask(query, k=4):
    chunks = retrieve(query, k)
    context = "\n\n".join(f"[page {c['page']}] {c['text']}" for c in chunks)
    prompt = f"""Answer the question using only the context below. Cite the page number(s) you used in square brackets, like [page 3]. If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {query}
Answer:"""
    answer = call_openai(prompt)
    sources = sorted(set(c["page"] for c in chunks))
    return answer, chunks, sources

In [ ]:
answer, chunks, sources = ask("what is the dual nature of light")
print(answer)
print("Pages used:", sources)

In [ ]:
!pip install -q rank_bm25
from rank_bm25 import BM25Okapi

tokenized_chunks = [c["text"].lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)

In [ ]:
def hybrid_retrieve(query, k=4, rrf_k=60):
    # FAISS ranking
    q_emb = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    _, faiss_idxs = index.search(q_emb, len(all_chunks))
    faiss_ranks = {idx: rank for rank, idx in enumerate(faiss_idxs[0])}

    # BM25 ranking
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ranks = {idx: rank for rank, idx in enumerate(np.argsort(-bm25_scores))}

    # Reciprocal Rank Fusion
    rrf_scores = {}
    for idx in range(len(all_chunks)):
        score = 1 / (rrf_k + faiss_ranks.get(idx, len(all_chunks))) + 1 / (rrf_k + bm25_ranks.get(idx, len(all_chunks)))
        rrf_scores[idx] = score

    top_idxs = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:k]
    return [all_chunks[i] for i in top_idxs]

In [ ]:
def ask_hybrid(query, k=4):
    chunks = hybrid_retrieve(query, k)
    context = "\n\n".join(f"[page {c['page']}] {c['text']}" for c in chunks)
    prompt = f"""Answer the question using only the context below. Cite the page number(s) you used in square brackets. If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {query}
Answer:"""
    answer = call_openai(prompt)
    return answer, chunks

for q in questions:
    answer, chunks = ask_hybrid(q)
    print("Q:", q)
    print("A:", answer)
    print("Pages:", [c["page"] for c in chunks])
    print("=" * 80)

In [ ]:
%%writefile ingest.py
import fitz, re

def extract_pages(pdf_path):
    doc = fitz.open(pdf_path)
    return [{"page": i + 1, "text": page.get_text()} for i, page in enumerate(doc)]

def clean_text(text, page_num, total_pages):
    text = text.replace(f"Page {page_num} of {total_pages}", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size]
        if chunk:
            chunks.append(" ".join(chunk))
        if i + chunk_size >= len(words):
            break
    return chunks

def build_chunks(pdf_path):
    pages = extract_pages(pdf_path)
    all_chunks = []
    chunk_id = 0
    for p in pages:
        clean = clean_text(p["text"], p["page"], len(pages))
        for c in chunk_text(clean):
            all_chunks.append({"chunk_id": chunk_id, "page": p["page"], "text": c})
            chunk_id += 1
    return all_chunks

In [ ]:
%%writefile retrieve.py
import numpy as np
import faiss
from rank_bm25 import BM25Okapi

def build_faiss_index(chunks, embedder):
    texts = [c["text"] for c in chunks]
    embeddings = embedder.encode(texts, show_progress_bar=True).astype("float32")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

def build_bm25_index(chunks):
    tokenized = [c["text"].lower().split() for c in chunks]
    return BM25Okapi(tokenized)

def hybrid_retrieve(query, chunks, index, bm25, embedder, k=4, rrf_k=60):
    q_emb = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    _, faiss_idxs = index.search(q_emb, len(chunks))
    faiss_ranks = {idx: rank for rank, idx in enumerate(faiss_idxs[0])}

    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ranks = {idx: rank for rank, idx in enumerate(np.argsort(-bm25_scores))}

    rrf_scores = {}
    for idx in range(len(chunks)):
        score = 1 / (rrf_k + faiss_ranks.get(idx, len(chunks))) + 1 / (rrf_k + bm25_ranks.get(idx, len(chunks)))
        rrf_scores[idx] = score

    top_idxs = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:k]
    return [chunks[i] for i in top_idxs]

In [ ]:
%%writefile rag.py
import os, requests

def call_openai(prompt, model="gpt-4o-mini"):
    r = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}", "Content-Type": "application/json"},
        json={"model": model, "messages": [{"role": "user", "content": prompt}], "temperature": 0}
    )
    data = r.json()
    if "error" in data:
        raise Exception(data["error"])
    return data["choices"][0]["message"]["content"]

def ask(query, chunks, retrieve_fn):
    retrieved = retrieve_fn(query)
    context = "\n\n".join(f"[page {c['page']}] {c['text']}" for c in retrieved)
    prompt = f"""Answer the question using only the context below. Cite the page number(s) in square brackets. If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {query}
Answer:"""
    return call_openai(prompt), retrieved

In [ ]:
%%writefile requirements.txt
pymupdf
sentence-transformers
faiss-cpu
rank_bm25
requests

In [ ]:
%%writefile .gitignore
*.index
*.json
*.pdf
__pycache__/